# Weakly-supervised crack segmentation

Segmenting cracks pixel by pixel while training on one bit per image: does this image contain a
crack or not.

Dataset: [Crack Segmentation Dataset](https://www.kaggle.com/datasets/lakshaymiddha/crack-segmentation-dataset), only the `train/` directory is used.

Jakub Laskowski (160287), Jakub Górniak (160326)

## What we are allowed to look at

| Stage | What it sees |
|---|---|
| Label creation | Each mask is opened once and turned into one bit: is there any crack pixel in this image. The mask is then thrown away. |
| Training (all stages) | Images and that one bit. |
| Evaluation | The real masks, on a held out split, after everything is frozen. |

No threshold and no hyperparameter is picked by looking at a mask.

## Why this is not a crack detector

Nothing here knows what a crack looks like. There are no edge detectors, no ridge filters, no
morphological thinning and no colour rules. We only assume that

1. a positive image has at least one positive pixel and a negative image has none,
2. the thing we are looking for covers a small part of the image,
3. its boundaries line up with edges in the image.

The same three things are true for "find the tumour cells given only healthy/sick slide labels", so
changing `DATA_ROOT` is enough to move the notebook to that problem.

## The three stages

```
       image + 1 bit                    coarse evidence                 full-resolution mask
    +----------------+              +--------------------+            +--------------------+
    |  1. MIL        |   score map  |  2. CAM -> pseudo  |  pseudo-   |  3. U-Net          |
    |  classifier    |------------->|     mask           |----------->|  distillation      |
    |  (ResNet-34)   |   56x56      |  TTA + guided      |  masks     |  (ResNet-34 enc.)  |
    +----------------+              |  filter + dual thr |            +--------------------+
                                    +--------------------+
```

1. MIL classifier. A fully convolutional ResNet-34 outputs a per pixel logit map and top-k pooling
   turns it into one logit for the whole image. Training only needs the bit, but the map in the
   middle is already a class activation map.
2. CAM to pseudo-mask. Multi-scale and flip TTA, a guided filter to sharpen the edges, then two
   thresholds: sure foreground, sure background, and an ignore band in between that we drop from the
   next loss.
3. Distillation. A U-Net is trained on the pseudo-masks. It averages out the CAM noise and gives
   full resolution masks. Then one round of self-training where the U-Net relabels the data, always
   forcing negative images to be completely background.

In [ ]:
import math
import os
import random
import re
import time
import warnings
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------- config
def find_data_root():
    """Kaggle does not always mount the dataset where we expect, so go and look for it."""
    if "DATA_ROOT" in os.environ:
        return Path(os.environ["DATA_ROOT"])
    mounted = Path("/kaggle/input")
    for depth in range(1, 7):
        pattern = "/".join(["*"] * depth) + "/train/images"
        for hit in sorted(mounted.glob(pattern)):
            if hit.is_dir():
                return hit.parent.parent
    return mounted / "crack-segmentation-dataset" / "crack_segmentation_dataset"


DATA_ROOT = find_data_root()
TRAIN_DIR = DATA_ROOT / "train"          # the only directory we are allowed to train on
WORK_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working"))

# these are the values behind the reported results, the env overrides are there so we could
# run the whole notebook quickly as a smoke test
IMG_SIZE = int(os.environ.get("IMG_SIZE", 448))          # classifier/segmenter input resolution
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", 8))
CLS_EPOCHS = int(os.environ.get("CLS_EPOCHS", 8))        # stage 1 - MIL classifier
SEG_EPOCHS = int(os.environ.get("SEG_EPOCHS", 10))       # stage 3 - U-Net, per self-training round
SELF_TRAINING_ROUNDS = int(os.environ.get("SELF_TRAINING_ROUNDS", 2))
SEED = 42

TOPK_RATIO = 0.02       # "the object covers ~2% of the image" prior, used by the MIL pooling
IGNORE_INDEX = 255      # pixels the pseudo-labeller is unsure about

FG_THRESHOLD = 0.45     # dual threshold on the normalised CAM
BG_THRESHOLD = 0.20
AREA_RATIO = TOPK_RATIO          # the foreground threshold keeps roughly this much of the image
RELABEL_AREA_CAP = 2 * TOPK_RATIO  # self-training is not allowed to grow the mask past this
GUIDED_RADIUS = 11      # the 8 we used at 320 px, scaled up with IMG_SIZE
TTA_SCALES = (0.5, 0.75, 1.0, 1.5)   # 1.5 x 448 already beats the 2.0 x 320 we used before

# both stages stop once the validation number stops moving, which is what keeps the run inside
# the session limit. the first run took 3.6 h and the classifier had already peaked by epoch 3
CLS_PATIENCE = int(os.environ.get("CLS_PATIENCE", 2))
SEG_PATIENCE = int(os.environ.get("SEG_PATIENCE", 3))
MIN_EPOCHS = int(os.environ.get("MIN_EPOCHS", 3))
PUZZLE_PROB = 0.5       # the puzzle branch costs a second forward pass, so only pay it half the time
W_CLDICE = 0.3          # topology term for the segmenter, see stage 3

# pseudo-masks are an intermediate, not a result. on kaggle everything under /kaggle/working is
# committed as the kernel output, and 20k pngs in there make the real artefacts impossible to pull
SCRATCH_DIR = Path(os.environ.get("SCRATCH_DIR", "/kaggle/temp" if Path("/kaggle").is_dir() else WORK_DIR))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sample_n(frame, n, seed=SEED):
    """Sample at most n rows so the figures still work on a small subset."""
    return frame.sample(min(n, len(frame)), random_state=seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True    # the input size never changes, so let cudnn pick its kernels once
RUN_STARTED = time.time()


def stage_done(name):
    """Print how far into the session we are, the notebook has to finish inside 12 h."""
    print(f"[{(time.time() - RUN_STARTED) / 60:6.1f} min] {name}")


print(f"device: {DEVICE} | torch {torch.__version__}")
print(f"data:   {TRAIN_DIR} (exists: {TRAIN_DIR.exists()})")
if not TRAIN_DIR.exists():
    for depth in range(1, 6):
        dirs = sorted(str(p) for p in Path("/kaggle/input").glob("/".join(["*"] * depth)) if p.is_dir())
        print(f"mounted, depth {depth}:", dirs[:10])


## 1. Data preparation, throwing the masks away

The cell below is the only place where a training mask is opened. Each one becomes
`label = int(mask.any())` and we cache the result in a CSV with just `image_path, label`. Everything
after that reads the CSV, so pixel information cannot leak into the training by accident.

The split is stratified on the label and on the source sub-dataset (the filename prefix, `CFD_`,
`DeepCrack_`, `Volker_` and so on). Those sub-datasets look very different, so a plain random split
would give us a validation set that is too easy.

In [ ]:
LABEL_CSV = WORK_DIR / "image_labels.csv"
WORK_DIR.mkdir(parents=True, exist_ok=True)


def source_group(stem: str) -> str:
    """Filename prefix, 'CFD_006' -> 'cfd'."""
    match = re.match(r"^([A-Za-z]+)", stem)
    return match.group(1).lower() if match else "unknown"


def build_label_table() -> pd.DataFrame:
    """Open every mask once, keep one bit per image and forget the pixels."""
    if LABEL_CSV.exists():
        return pd.read_csv(LABEL_CSV)

    image_paths = sorted((TRAIN_DIR / "images").glob("*"))
    mask_dir = TRAIN_DIR / "masks"
    rows = []
    for path in tqdm(image_paths, desc="deriving image-level labels"):
        mask_path = next(mask_dir.glob(path.stem + ".*"))
        mask = np.asarray(Image.open(mask_path).convert("L"))
        rows.append(
            {
                "image_path": str(path),
                "stem": path.stem,
                "group": source_group(path.stem),
                "label": int((mask > 127).any()),   # the one bit we keep
            }
        )
    table = pd.DataFrame(rows)
    table.to_csv(LABEL_CSV, index=False)
    return table


table = build_label_table()
assert len(table), f"no images under {TRAIN_DIR / 'images'}, check the dataset is attached"

# stratify on (label, source) so every split sees every imaging condition. a group needs about
# eight images to still leave two in the holdout, smaller ones join the biggest group that
# has the same label
labels = table.label.astype(str)
strata = labels + "|" + table.group
counts = strata.value_counts()
biggest = {lab: counts[counts.index.str.startswith(lab + "|")].index[0] for lab in labels.unique()}
strata = strata.mask(strata.map(counts) < 8, labels.map(biggest))

train_idx, holdout_idx = train_test_split(np.arange(len(table)), test_size=0.25, random_state=SEED, stratify=strata)
val_idx, test_idx = train_test_split(holdout_idx, test_size=0.6, random_state=SEED, stratify=strata.iloc[holdout_idx])

table["split"] = "train"
table.loc[val_idx, "split"] = "val"
table.loc[test_idx, "split"] = "test"

train_df = table[table.split == "train"].reset_index(drop=True)
val_df = table[table.split == "val"].reset_index(drop=True)
test_df = table[table.split == "test"].reset_index(drop=True)

print(f"total images: {len(table)}")
print(table.groupby("split").agg(n=("label", "size"), positives=("label", "sum")))


In [ ]:
# EDA on what we are allowed to see: the images and their one bit.
counts = table.groupby(["group", "label"]).size().unstack(fill_value=0)
counts = counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]

fig = plt.figure(figsize=(14, 4.5))
gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 1])

ax = fig.add_subplot(gs[0])
counts.plot(kind="bar", stacked=True, ax=ax, color=["#9ecae1", "#e6550d"], width=0.8)
ax.set_title("Images per source sub-dataset")
ax.set_xlabel("")
ax.set_ylabel("images")
ax.legend(["no crack", "crack"])
ax.tick_params(axis="x", rotation=45)

ax = fig.add_subplot(gs[1])
overall = table.label.value_counts().sort_index()
ax.bar(["no crack", "crack"], overall.values, color=["#9ecae1", "#e6550d"])
for i, v in enumerate(overall.values):
    ax.text(i, v, f"{v}\n({v / len(table):.0%})", ha="center", va="bottom")
ax.set_title("Overall class balance")
ax.set_ylim(0, overall.max() * 1.2)
plt.tight_layout()
plt.show()

sizes = {Image.open(p).size for p in sample_n(table, 50).image_path}
print(f"image sizes in a random sample: {sizes}")

sample = pd.concat([sample_n(table[table.label == 1], 6), sample_n(table[table.label == 0], 6)])
fig, axes = plt.subplots(2, 6, figsize=(15, 5.2))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows(), strict=False):
    ax.imshow(Image.open(row.image_path))
    ax.set_title(f"{'crack' if row.label else 'no crack'} · {row.group}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("What the model gets to see during training", y=1.0)
plt.tight_layout()
plt.show()


### Preprocessing and augmentation

Random resized crop to 448x448 (we started at 320 and the extra resolution matters, a crack is only
a few pixels wide), ImageNet normalisation, and augmentations that do not change the label and do
not depend on the orientation: flips and 90 degree rotations (a crack upside down is still a crack)
and small affine warps.

The photometric half of the list is heavier than the task needs on purpose. 87% of the images
contain a crack and roughly 96% of the crack free ones come from a single source sub-dataset, so
"is this image from `noncrack`?" is very nearly the right answer to "is this image crack free?".
Grayscale, gamma, strong brightness/contrast and JPEG artefacts are there to make the source harder
to recognise than the crack. For the same reason the training loader draws from a
`WeightedRandomSampler` so both classes turn up equally often instead of 87/13.

`CrackDataset` returns the image and the label and never stores a mask path, so the rule is enforced
by the code and not by us remembering it.

In [ ]:
# almost every crack free image comes from one source sub-dataset, so the classifier could get a
# good score by recognising the sub-dataset instead of the crack. the photometric augmentations are
# heavier than they need to be for the task on purpose, they are there to take that shortcut away
train_tf = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.5, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.8, 1.25), rotate=(-20, 20), p=0.5),
    A.ToGray(p=0.3),
    A.RandomBrightnessContrast(0.35, 0.35, p=0.8),
    A.RandomGamma(gamma_limit=(70, 140), p=0.5),
    A.HueSaturationValue(8, 20, 10, p=0.3),
    A.ImageCompression(quality_range=(40, 95), p=0.3),
    A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.MotionBlur(blur_limit=5)], p=0.2),
    A.GaussNoise(p=0.3),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ToTensorV2(),
])

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ToTensorV2(),
])


def denormalise(tensor):
    """CHW normalised tensor -> HWC image in [0, 1], for plotting."""
    array = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(array * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)


class CrackDataset(Dataset):
    """Images and one binary label, there is no mask anywhere in here."""

    def __init__(self, frame, transform, with_raw=False):
        self.paths = frame.image_path.tolist()
        self.labels = frame.label.astype(np.float32).tolist()
        self.stems = frame.stem.tolist()
        self.transform = transform
        self.with_raw = with_raw

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        raw = np.asarray(Image.open(self.paths[idx]).convert("RGB"))
        item = {"image": self.transform(image=raw)["image"], "label": self.labels[idx], "stem": self.stems[idx]}
        if self.with_raw:
            item["raw"] = cv2.resize(raw, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return item


def make_loader(frame, transform, shuffle=False, batch_size=BATCH_SIZE, with_raw=False, sampler=None):
    return DataLoader(
        CrackDataset(frame, transform, with_raw),
        batch_size=batch_size,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        drop_last=shuffle,
    )


def balanced_sampler(frame):
    """Only 13% of the images are crack free, so draw both classes equally often."""
    counts = frame.label.value_counts()
    weights = frame.label.map(1.0 / counts).to_numpy()
    return torch.utils.data.WeightedRandomSampler(weights, num_samples=len(frame), replacement=True)


train_loader = make_loader(train_df, train_tf, shuffle=True, sampler=balanced_sampler(train_df))
val_loader = make_loader(val_df, eval_tf)
print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)}")


## 2. Stage 1, a classifier that also gives us a map

### Picking the architecture

The image is a bag and every position of the feature map is an instance. The label says either "at
least one instance is positive" or "all of them are negative", which is multiple instance learning.
It is the same setup as a slide that is positive if it contains at least one tumour cell.

So we build the network as a segmenter and only collapse it to one number at the end:

```
image -> ResNet-34 (ImageNet, dilated) -> 3x3 conv + BN + ReLU -> 1x1 conv -> score map S (B,1,40,40)
                                                                                  |
                                                                            top-k pooling
                                                                                  v
                                                                            image logit (B,1)
```

| Choice | Alternative | Why |
|---|---|---|
| Dilated ResNet-34 (no stride in `layer3`/`layer4`, dilation 2 and 4, output stride 8) | plain ResNet-50 at stride 32 | Cracks are a few pixels wide. Stride 32 gives a 14x14 map which is useless. Dilation gives 56x56 with the same parameters and the same receptive field, and ResNet-34 keeps the 4x compute increase manageable. `replace_stride_with_dilation` only works on `Bottleneck` nets so we had to do it by hand. |
| Top-k pooling, k = 2% of the positions | global average pooling | GAP averages 3136 positions, so a 40 pixel crack is well under 2% of the logit and the gradient drowns. Top-k is a soft maximum and its k says "the object is small". Global max pooling is the other extreme, correct in principle but a single position gives a very noisy gradient. |
| No bias on the 1x1 conv | with a bias | Without it the image logit stays a pooled linear projection of the features, which is what makes S an actual CAM and not an approximation of one. |

There is also an auxiliary max pooling head on the same score map. It says that on a negative image
no position is allowed to fire, which is how the negative images give us real supervision for the
background.

In [ ]:
def dilate_stage(stage, dilation):
    """Turn a stride-2 ResNet stage into a stride-1 dilated stage (keeps the receptive field)."""
    for module in stage.modules():
        if isinstance(module, nn.Conv2d):
            if module.stride == (2, 2):
                module.stride = (1, 1)
            if module.kernel_size == (3, 3):
                module.dilation = (dilation, dilation)
                module.padding = (dilation, dilation)


class MILClassifier(nn.Module):
    """Fully-convolutional ResNet-34 + 1x1 classifier + top-k MIL pooling."""

    def __init__(self, pretrained=True, topk_ratio=TOPK_RATIO):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = torchvision.models.resnet34(weights=weights)
        # torchvision only does replace_stride_with_dilation for Bottleneck nets,
        # so we do it by hand. Output stride goes from 32 to 8.
        dilate_stage(backbone.layer3, 2)
        dilate_stage(backbone.layer4, 4)

        self.encoder = nn.Sequential(*list(backbone.children())[:-2])   # -> (B, 512, H/8, W/8)
        self.neck = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
        )
        self.classifier = nn.Conv2d(256, 1, 1, bias=False)   # no bias, otherwise it is not a real CAM
        self.topk_ratio = topk_ratio

    def score_map(self, x):
        """Per-pixel logits at input_size / 8."""
        return self.classifier(self.neck(self.encoder(x)))

    def topk_pool(self, score):
        flat = score.flatten(2)                                   # (B, 1, H*W)
        k = max(1, round(self.topk_ratio * flat.shape[-1]))
        return flat.topk(k, dim=-1).values.mean(-1)               # (B, 1)

    @staticmethod
    def max_pool(score):
        return score.flatten(2).max(-1).values                    # (B, 1)

    def forward(self, x):
        score = self.score_map(x)
        return {"logit": self.topk_pool(score), "max_logit": self.max_pool(score), "score_map": score}


def normalise_map(maps, eps=1e-6):
    """Per-image min-max scaling of an activation map to [0, 1]."""
    flat = maps.flatten(2)
    lo = flat.min(-1, keepdim=True).values
    hi = flat.max(-1, keepdim=True).values
    return ((flat - lo) / (hi - lo + eps)).view_as(maps).clamp(0, 1)


model = MILClassifier().to(DEVICE)
with torch.no_grad():
    out = model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
print(f"score map: {tuple(out['score_map'].shape)}   image logit: {tuple(out['logit'].shape)}")
print(f"trainable parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"top-k pooling uses k = {max(1, round(TOPK_RATIO * out['score_map'][0, 0].numel()))} of {out['score_map'][0, 0].numel()} positions")


### The loss: one bit plus self-consistency

$$\mathcal{L} = \underbrace{\mathrm{BCE}(\mathrm{topk}(S), y)}_{\text{bag}} + 0.4\,\underbrace{\mathrm{BCE}(\max(S), y)}_{\text{strict MIL}} + \underbrace{\mathrm{BCE}(\mathrm{topk}(\tilde S), y) + \lambda_p \lVert S - \tilde S \rVert_1}_{\text{puzzle consistency}} + \lambda_s\,\overline{\sigma(S)}\big|_{y=1}$$

The usual problem with CAM based segmentation is that a classifier only needs the most obvious part
of the object to get the label right, so the map lights up a small blob and stops there. Two extra
terms deal with that and neither of them needs more than the bit we already have.

* Puzzle consistency (Puzzle-CAM). Cut the image into 2x2 tiles, classify each tile on its own and
  stitch the four maps back together into $\tilde S$. A tile cannot use evidence that lives in
  another tile, so the network has to respond everywhere the evidence is, and $\tilde S$ has to
  agree with the whole image map $S$. This was the single most useful term we tried.
* Foreground area prior ($\lambda_s = 0.02$, positives only), a small push towards a smaller
  activated area. This is assumption 2, set it to 0 if the target is large.

The regularisers are warmed up linearly over the first two epochs. Before that $S$ is basically
noise and forcing it to be consistent with itself only slows the training down.

The puzzle term needs a second forward pass over the tiles and that is most of the cost of a step,
so we only apply it on a random half of the steps. That halves its strength as well as its price,
which is a fair trade here: the classifier reaches validation AUROC 1.0 either way and the budget
is better spent on the segmenter. Training stops as soon as validation AUROC has been flat for two
epochs, which on the full dataset saves about a third of stage 1 for no loss at all.

In [ ]:
W_MAX, W_PUZZLE, W_SPARSITY = 0.4, 0.5, 0.02
REG_WARMUP_EPOCHS = 2


def tile(x, n=2):
    """(B,C,H,W) -> (B*n*n, C, H/n, W/n), row-major tile order."""
    b, c, h, w = x.shape
    x = x.reshape(b, c, n, h // n, n, w // n).permute(0, 2, 4, 1, 3, 5)
    return x.reshape(b * n * n, c, h // n, w // n)


def untile(x, batch, n=2):
    """Inverse of `tile`."""
    _, c, th, tw = x.shape
    x = x.reshape(batch, n, n, c, th, tw).permute(0, 3, 1, 4, 2, 5)
    return x.reshape(batch, c, n * th, n * tw)


def mil_loss(model, images, labels, reg_scale=1.0):
    out = model(images)
    terms = {
        "bag": F.binary_cross_entropy_with_logits(out["logit"].squeeze(1), labels),
        "strict": W_MAX * F.binary_cross_entropy_with_logits(out["max_logit"].squeeze(1), labels),
    }

    # the puzzle branch needs a second forward pass over the tiles, which is most of the cost of a
    # step. sampling it halves that for half the regularisation, and stage 1 reaches AUROC 1.0 either way
    if reg_scale > 0 and W_PUZZLE > 0 and random.random() < PUZZLE_PROB:
        tiled = untile(model.score_map(tile(images)), images.shape[0])
        terms["puzzle_cls"] = F.binary_cross_entropy_with_logits(model.topk_pool(tiled).squeeze(1), labels)
        terms["puzzle_l1"] = reg_scale * W_PUZZLE * F.l1_loss(out["score_map"], tiled)

    if reg_scale > 0 and W_SPARSITY > 0 and (labels > 0.5).any():
        terms["sparsity"] = reg_scale * W_SPARSITY * torch.sigmoid(out["score_map"][labels > 0.5]).mean()

    total = torch.stack(list(terms.values())).sum()
    return total, {k: float(v.detach()) for k, v in terms.items()} | {"total": float(total.detach())}


In [ ]:
CLS_CKPT = WORK_DIR / "mil_classifier.pth"


def cosine_with_warmup(optimizer, total_steps, warmup=200):
    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(total_steps - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def classifier_scores(model, loader):
    """Image level probabilities and labels, no masks involved."""
    model.eval()
    scores, labels = [], []
    for batch in loader:
        logit = model(batch["image"].to(DEVICE))["logit"].squeeze(1)
        scores.append(torch.sigmoid(logit).float().cpu().numpy())
        labels.append(batch["label"].numpy())
    return np.concatenate(scores), np.concatenate(labels)


def train_classifier(model, epochs=CLS_EPOCHS, patience=CLS_PATIENCE, min_epochs=MIN_EPOCHS):
    head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder")]
    optimizer = torch.optim.AdamW(
        [{"params": model.encoder.parameters(), "lr": 3e-5}, {"params": head_params, "lr": 3e-4}],
        weight_decay=1e-4,
    )
    scheduler = cosine_with_warmup(optimizer, epochs * len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    history, best_auc, stale = [], -1.0, 0

    for epoch in range(epochs):
        model.train()
        reg_scale = min(1.0, epoch / REG_WARMUP_EPOCHS) if REG_WARMUP_EPOCHS else 1.0
        running, started = {}, time.time()

        for batch in tqdm(train_loader, desc=f"epoch {epoch + 1}/{epochs}", leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss, report = mil_loss(model, images, labels, reg_scale)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            for k, v in report.items():
                running[k] = running.get(k, 0.0) + v

        train_stats = {k: v / len(train_loader) for k, v in running.items()}
        scores, labels = classifier_scores(model, val_loader)
        val_auc = roc_auc_score(labels, scores)
        val_acc = ((scores >= 0.5) == labels).mean()
        history.append({"epoch": epoch + 1, **train_stats, "val_auroc": val_auc, "val_acc": val_acc})
        print(f"epoch {epoch + 1:2d}  loss {train_stats['total']:.4f}  val AUROC {val_auc:.4f}  val acc {val_acc:.4f}  ({time.time() - started:.0f}s)")

        if val_auc > best_auc + 1e-4:               # selection uses image-level labels only
            best_auc, stale = val_auc, 0
            torch.save(model.state_dict(), CLS_CKPT)
        else:
            stale += 1

        if epoch + 1 >= min_epochs and stale >= patience:
            print(f"val AUROC has not improved for {patience} epochs, stopping")
            break
    return pd.DataFrame(history)


set_seed()
cls_history = train_classifier(model)
model.load_state_dict(torch.load(CLS_CKPT, map_location=DEVICE))
print(f"best val AUROC: {cls_history.val_auroc.max():.4f}")
stage_done("stage 1, MIL classifier")


In [ ]:
# the gate decides whether an image gets a mask at all. an empty prediction scores dice 1 on an
# image with no crack and dice 0 on one with a crack, so the threshold is worth choosing rather than
# leaving at 0.5. we pick it on validation using the image level labels, which is all we are allowed
val_scores, val_labels = classifier_scores(model, val_loader)
candidates = np.linspace(0.05, 0.95, 19)
gate_f1 = []
for threshold in candidates:
    predicted = val_scores >= threshold
    hits = float((predicted & (val_labels > 0.5)).sum())
    gate_f1.append(2 * hits / max(float(predicted.sum()) + float((val_labels > 0.5).sum()), 1e-9))
GATE_THRESHOLD = float(candidates[int(np.argmax(gate_f1))])
print(f"gate threshold picked on val: {GATE_THRESHOLD:.2f} (val F1 {max(gate_f1):.4f})")

test_loader = make_loader(test_df, eval_tf)
test_scores, test_labels = classifier_scores(model, test_loader)
test_pred = (test_scores >= GATE_THRESHOLD).astype(int)

tp = int(((test_pred == 1) & (test_labels == 1)).sum())
fp = int(((test_pred == 1) & (test_labels == 0)).sum())
fn = int(((test_pred == 0) & (test_labels == 1)).sum())
tn = int(((test_pred == 0) & (test_labels == 0)).sum())
precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)

cls_metrics = {
    "accuracy": (tp + tn) / len(test_labels),
    "precision": precision,
    "recall": recall,
    "f1": 2 * precision * recall / max(precision + recall, 1e-9),
    "auroc": roc_auc_score(test_labels, test_scores),
    "average_precision": average_precision_score(test_labels, test_scores),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
loss_cols = [c for c in cls_history.columns if c not in {"epoch", "val_auroc", "val_acc", "total"}]
for col in loss_cols:
    axes[0].plot(cls_history.epoch, cls_history[col], label=col, alpha=0.8)
axes[0].plot(cls_history.epoch, cls_history.total, "k--", lw=2, label="total")
axes[0].set_title("Training loss terms")
axes[0].set_xlabel("epoch")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(cls_history.epoch, cls_history.val_auroc, "o-", label="val AUROC")
axes[1].plot(cls_history.epoch, cls_history.val_acc, "s-", label="val accuracy")
axes[1].set_title("Image-level validation")
axes[1].set_xlabel("epoch")
axes[1].set_ylim(0.5, 1.02)
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].imshow([[tn, fp], [fn, tp]], cmap="Blues")
for (i, j), v in np.ndenumerate([[tn, fp], [fn, tp]]):
    axes[2].text(j, i, str(v), ha="center", va="center", fontsize=14)
axes[2].set_xticks([0, 1], ["pred no crack", "pred crack"])
axes[2].set_yticks([0, 1], ["no crack", "crack"])
axes[2].set_title("Test confusion matrix")
plt.tight_layout()
plt.show()

print("image-level test metrics:")
for k, v in cls_metrics.items():
    print(f"  {k:18s} {v:.4f}")

# 87% of the images contain a crack, so accuracy alone looks good no matter what. average precision
# and the per source breakdown are the numbers that actually say whether the classifier learned the
# crack or just learned which sub-dataset the image came from
per_source = test_df.assign(correct=(test_pred == test_labels.astype(int)))
summary = per_source.groupby("group").agg(
    n=("label", "size"), positives=("label", "sum"), accuracy=("correct", "mean"))
print("\nper source accuracy:")
print(summary.sort_values("n", ascending=False).round(4).to_string())

# most of the sources are 100% cracked and nearly every crack-free image comes from one of them,
# so "which sub-dataset is this" already answers "is there a crack" almost perfectly. the readout
# below throws away the sources that only ever show one class, which is the part of the image-level
# score that the shortcut cannot explain. it is a small sample and we report it as such
mixed = summary[(summary.positives > 0) & (summary.positives < summary.n)].index
inside = test_df.group.isin(mixed).to_numpy()
if inside.sum() and 0 < test_labels[inside].sum() < inside.sum():
    positives_inside = int(test_labels[inside].sum())
    print(f"\nsources containing both classes ({list(mixed)}):")
    print(f"  n {int(inside.sum())} ({positives_inside} cracked / {int(inside.sum()) - positives_inside} not)")
    print(f"  auroc              {roc_auc_score(test_labels[inside], test_scores[inside]):.4f}")
    print(f"  average_precision  {average_precision_score(test_labels[inside], test_scores[inside]):.4f}")
else:
    print("\nno source contains both classes, the image-level score cannot be separated from the shortcut")
stage_done("stage 1 evaluation")


## 3. Stage 2, from a 56x56 score map to pixels

Three generic steps, in this order:

1. Multi-scale and flip TTA. We recompute the score map at scales 0.5 / 0.75 / 1.0 / 1.5 with
   horizontal and vertical flips and average the 12 passes. The small scales give context and the
   1.5 scale gives resolution (at 672 px input the map is effectively 84x84). The scales used to go
   up to 2.0, but that was there to make up for a 320 px base, and 1.5 x 448 is already finer than
   2.0 x 320 for less than half the compute. Averaging also cancels
   part of the orientation bias that a stack of convolutions always has.
2. Guided filter. The CAM is smooth and limited by the stride, but the image itself already knows
   where the edges are, so a guided filter (He et al.) copies that structure onto the probability
   map. This is assumption 3 and it is not task specific, it is the same filter you would use to
   snap a tumour heat map to the cell boundaries.
3. Two thresholds. Instead of deciding everywhere, pixels above `fg` become foreground, pixels below
   `bg` become background and everything in between becomes `ignore` and is dropped from the stage 3
   loss. Letting the pseudo-labeller say "I don't know" is what stops the U-Net from copying the
   CAM's mistakes. The foreground threshold is a per image quantile that keeps `TOPK_RATIO` of the
   pixels, the same 2% object size prior the MIL pooling already uses. Otsu was the obvious choice and it was the wrong one: it splits the histogram into two
   classes of similar size, while a crack is around 2% of the image, so it cut too low and the
   pseudo-masks came out much thicker than the crack. On a synthetic crack (a 1-3 px line on a
   noisy background, so we know the answer) Otsu gets 0.10 IoU against 0.25-0.33 for the quantile.
   Like Otsu, the quantile is computed from the CAM alone, never from a mask.

On top of that there is the image level gate: if the classifier says there is no crack, the whole
pseudo-mask is background. That is the only exact supervision we have and we apply it at every
stage, self-training included.

In [ ]:
@torch.no_grad()
def multi_scale_cam(model, images, scales=TTA_SCALES, flips=True, size=(IMG_SIZE, IMG_SIZE)):
    """Scale/flip-averaged score map, min-max normalised per image to [0, 1]."""
    model.eval()
    accumulator = torch.zeros(images.shape[0], 1, *size, device=images.device)
    variants = [(), (-1,), (-2,)] if flips else [()]

    for scale in scales:
        scaled = images if scale == 1.0 else F.interpolate(
            images, scale_factor=scale, mode="bilinear", align_corners=False)
        for dims in variants:
            batch = torch.flip(scaled, dims) if dims else scaled
            score = model.score_map(batch)
            if dims:
                score = torch.flip(score, dims)
            accumulator += F.interpolate(F.relu(score), size=size, mode="bilinear", align_corners=False)

    return normalise_map(accumulator / (len(scales) * len(variants)))


def guided_filter(guide_rgb, source, radius=GUIDED_RADIUS, eps=1e-3):
    """Edge preserving filter, pulls source onto the edges of guide_rgb."""
    guide = cv2.cvtColor(guide_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    source = source.astype(np.float32)
    ksize = (2 * radius + 1, 2 * radius + 1)

    def box(x):
        return cv2.boxFilter(x, -1, ksize, normalize=True, borderType=cv2.BORDER_REFLECT)

    mean_g, mean_s = box(guide), box(source)
    var_g = box(guide * guide) - mean_g * mean_g
    cov_gs = box(guide * source) - mean_g * mean_s
    a = cov_gs / (var_g + eps)
    b = mean_s - a * mean_g
    return np.clip(box(a) * guide + box(b), 0, 1)


def rescale(x, eps=1e-6):
    lo, hi = float(x.min()), float(x.max())
    return (x - lo) / (hi - lo + eps)


def area_threshold(probability, ratio=AREA_RATIO):
    """Per image threshold that keeps about `ratio` of the pixels, using the object size prior."""
    return float(np.quantile(probability, 1.0 - ratio))


def probability_to_pseudo_label(probability, is_positive):
    """0 = background, 1 = foreground, IGNORE_INDEX = unknown."""
    if not is_positive:
        return np.zeros(probability.shape, np.uint8)          # no crack in the image, so no crack pixels
    # otsu was the obvious choice here and it was the wrong one: it splits the histogram into two
    # classes of similar size, while a crack covers about 2% of the image, so it always cut too low
    # and the pseudo-mask came out several times thicker than the crack. the quantile uses the same
    # size prior as the MIL pooling and it bounds the area by construction
    fg = area_threshold(probability)
    bg = min(BG_THRESHOLD, 0.75 * fg)
    label = np.full(probability.shape, IGNORE_INDEX, np.uint8)
    label[probability < bg] = 0
    label[probability >= fg] = 1
    return label


def refined_probability(cam_slice, raw_image):
    """CAM -> refined, renormalised probability map at IMG_SIZE."""
    return rescale(guided_filter(raw_image, cam_slice))


In [ ]:
# the whole point of the project is that no mask is ever used for training, and that is easy to
# break by accident. every read goes through here and gets counted, and the counter for the
# training split is asserted to be zero right before the results are computed
TRAIN_STEMS = set(train_df.stem)
GT_READS = {"train": 0, "held out": 0}


def load_gt_mask(stem, size=None):
    """Ground truth, only for the figures and the scoring, never for training."""
    GT_READS["train" if stem in TRAIN_STEMS else "held out"] += 1
    path = next((TRAIN_DIR / "masks").glob(stem + ".*"))
    mask = (np.asarray(Image.open(path).convert("L")) > 127).astype(np.uint8)
    return cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST) if size else mask


# walk a few positive test images through every step of stage 2
demo_df = sample_n(test_df[test_df.label == 1], 4).reset_index(drop=True)
demo_batch = next(iter(make_loader(demo_df, eval_tf, batch_size=len(demo_df), with_raw=True)))
demo_images = demo_batch["image"].to(DEVICE)

single_cam = multi_scale_cam(model, demo_images, scales=(1.0,), flips=False)[:, 0].cpu().numpy()
tta_cam = multi_scale_cam(model, demo_images)[:, 0].cpu().numpy()

titles = ["image", "CAM (single scale)", "CAM + TTA", "+ guided filter", "pseudo-mask", "ground truth"]
fig, axes = plt.subplots(len(demo_df), 6, figsize=(17, 2.9 * len(demo_df)), squeeze=False)
for i in range(len(demo_df)):
    raw = demo_batch["raw"][i].numpy()
    refined = refined_probability(tta_cam[i], raw)
    pseudo = probability_to_pseudo_label(refined, is_positive=True)
    truth = load_gt_mask(demo_batch["stem"][i], (IMG_SIZE, IMG_SIZE))

    for j, panel in enumerate([raw, single_cam[i], tta_cam[i], refined, pseudo, truth]):
        ax = axes[i][j]
        if j == 0:
            ax.imshow(panel)
        elif j == 4:
            ax.imshow(np.where(panel == IGNORE_INDEX, 0.5, panel), cmap="gray", vmin=0, vmax=1)
        elif j == 5:
            ax.imshow(panel, cmap="gray", vmin=0, vmax=1)
        else:
            ax.imshow(panel, cmap="inferno", vmin=0, vmax=1)
        if i == 0:
            ax.set_title(titles[j], fontsize=11)
        ax.axis("off")
fig.text(0.63, 0.005, "pseudo-mask: black = background, grey = ignore, white = foreground", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
PSEUDO_DIR = SCRATCH_DIR / "pseudo"


@torch.no_grad()
def generate_pseudo_masks(model, frame, out_dir):
    """Write one uint8 PNG per image: {0 background, 1 foreground, 255 ignore}."""
    out_dir.mkdir(parents=True, exist_ok=True)
    loader = make_loader(frame, eval_tf, with_raw=True)
    composition = {"foreground": 0.0, "background": 0.0, "ignore": 0.0}

    for batch in tqdm(loader, desc=f"pseudo-labelling -> {out_dir.name}", leave=False):
        cams = multi_scale_cam(model, batch["image"].to(DEVICE))[:, 0].cpu().numpy()
        for i, stem in enumerate(batch["stem"]):
            refined = refined_probability(cams[i], batch["raw"][i].numpy())
            # train/val images come with their image-level bit, so the gate is exact here
            label = probability_to_pseudo_label(refined, bool(batch["label"][i] > 0.5))
            Image.fromarray(label).save(out_dir / f"{stem}.png")
            composition["foreground"] += float((label == 1).mean())
            composition["background"] += float((label == 0).mean())
            composition["ignore"] += float((label == IGNORE_INDEX).mean())

    return {k: v / len(frame) for k, v in composition.items()}


set_seed()
pseudo_stats = {split: generate_pseudo_masks(model, frame, PSEUDO_DIR / split)
                for split, frame in [("train", train_df), ("val", val_df)]}

print("average pseudo-mask composition")
print(pd.DataFrame(pseudo_stats).T.round(4))
stage_done("stage 2, pseudo-labelling")


## 4. Stage 3, distilling the pseudo-masks into a U-Net

A CAM is a low resolution detector, a U-Net with skip connections is what actually draws the
outline. Training the U-Net on the confident parts of the pseudo-masks gives us three things:

* Resolution. The skip connections carry the stride 2 and stride 4 detail that the classifier's
  stride 8 trunk had thrown away, so the output is really full resolution.
* Denoising. The CAM errors are mostly independent between images, so a network that has to explain
  thousands of them with one set of weights fits the part that is consistent and averages the rest
  away. The student ends up better than the teacher.
* Room to disagree. Around a tenth of the pixels sit in the ignore band and have no target at all,
  so the network is not punished for correcting the teacher where the teacher was unsure.

Loss: partial BCE, a Tversky term with `alpha = 0.3, beta = 0.7`, and a clDice term, all three only
on the pixels that are not ignored.

* Tversky charges more for a false positive than for a miss. The first version used `pos_weight = 4`
  and plain Dice, which does the opposite: it buys recall with precision, and the predictions were
  already too thick, so it made the main failure mode worse.
* Tversky controls how wide the crack comes out but says nothing about whether it stays in one
  piece, and in the first full run the U-Net scored worse on boundary F1 than the map it had learned
  from. clDice compares centrelines instead of areas: it skeletonises both masks with a
  differentiable erode/dilate and asks whether each skeleton lies inside the other mask, so a crack
  broken into fragments loses even when the overlap is fine. On its own it would not notice a mask
  that is too thick, which is exactly the half Tversky already covers, so the two are complementary.

Self-training: after round 1 the U-Net relabels the training set itself, `p >= 0.7` is foreground,
`p <= 0.3` is background, the rest is ignore, and negative images are forced to background. The
foreground threshold is also capped by area so one round cannot mark more than about 4% of an image,
otherwise every round hands the next one a slightly thicker crack. Round 2 trains on those.

Picking the model we ship needs care here. Every round is trained against its own pseudo-masks, so
the validation losses of round 1 and round 2 are computed against different targets and comparing
them is meaningless. In the first full run round 2 looked worse purely for that reason and still
overwrote the checkpoint. Now each round is additionally scored against the fixed stage-2 masks,
which gives every round one number on the same scale, and the best of those is the model that gets
used. A later round is allowed to be worse and if it is, we keep the earlier one. All of this is
still pseudo-masks, so no real mask is involved.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.block(x)


class UNet(nn.Module):
    """Classic U-Net with an ImageNet-pretrained ResNet-34 encoder."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = torchvision.models.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)   # /2,  64
        self.pool, self.layer1 = backbone.maxpool, backbone.layer1               # /4,  64
        self.layer2, self.layer3, self.layer4 = backbone.layer2, backbone.layer3, backbone.layer4
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.dec0 = DecoderBlock(32, 0, 16)
        self.head = nn.Conv2d(16, 1, 3, padding=1)

    def forward(self, x):
        f0 = self.stem(x)
        f1 = self.layer1(self.pool(f0))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        d = self.dec4(f4, f3)
        d = self.dec3(d, f2)
        d = self.dec2(d, f1)
        d = self.dec1(d, f0)
        return self.head(self.dec0(d))


class PseudoMaskDataset(Dataset):
    """Image plus the mask we generated ourselves."""

    def __init__(self, frame, pseudo_dir, transform):
        self.paths = frame.image_path.tolist()
        self.stems = frame.stem.tolist()
        self.pseudo_dir = Path(pseudo_dir)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = np.asarray(Image.open(self.paths[idx]).convert("RGB"))
        pseudo = np.asarray(Image.open(self.pseudo_dir / f"{self.stems[idx]}.png"))
        # pseudo-masks live at IMG_SIZE, the images at their native size
        image = cv2.resize(image, (pseudo.shape[1], pseudo.shape[0]), interpolation=cv2.INTER_LINEAR)
        augmented = self.transform(image=image, mask=pseudo)
        return {"image": augmented["image"], "target": augmented["mask"].long()}


# only flips and rotations, an affine warp would have to invent border pixels and we
# would not know whether to mark them as ignore
seg_train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(0.25, 0.25, p=0.5),
    A.HueSaturationValue(8, 20, 10, p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ToTensorV2(),
])
seg_eval_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])


def soft_erode(x):
    return -F.max_pool2d(-x, 3, stride=1, padding=1)


def soft_open(x):
    return F.max_pool2d(soft_erode(x), 3, stride=1, padding=1)


def soft_skeleton(x, iters=5):
    """Differentiable morphological thinning, the soft skeleton from the clDice paper."""
    opened = soft_open(x)
    skeleton = F.relu(x - opened)
    for _ in range(iters):
        x = soft_erode(x)
        delta = F.relu(x - soft_open(x))
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


def cl_dice(probability, reference, eps=1.0):
    """Dice between each mask and the other one's centreline, so a broken crack is punished."""
    pred_skeleton, true_skeleton = soft_skeleton(probability), soft_skeleton(reference)
    dims = (1, 2, 3)
    precision = ((pred_skeleton * reference).sum(dims) + eps) / (pred_skeleton.sum(dims) + eps)
    sensitivity = ((true_skeleton * probability).sum(dims) + eps) / (true_skeleton.sum(dims) + eps)
    return (2 * precision * sensitivity / (precision + sensitivity)).mean()


def pseudo_loss(logits, target, pos_weight=1.0, alpha=0.3, beta=0.7):
    """Partial BCE + Tversky + clDice, evaluated only where the pseudo-labeller was confident."""
    logits = logits.squeeze(1)
    valid = target != IGNORE_INDEX
    if valid.sum() == 0:
        return logits.sum() * 0.0
    weight = torch.tensor(pos_weight, device=logits.device)
    ce = F.binary_cross_entropy_with_logits(logits[valid], target[valid].float(), pos_weight=weight)

    # dice is symmetric and pos_weight 4 was actively buying recall with precision, which is exactly
    # the wrong trade here - the predictions were already too thick. tversky with beta > alpha
    # charges more for a false positive than for a miss and pulls the crack back to its real width
    probability = torch.sigmoid(logits) * valid
    reference = (target == 1).float()
    true_pos = (probability * reference).sum((1, 2))
    false_neg = ((1 - probability) * reference).sum((1, 2))
    false_pos = (probability * (1 - reference)).sum((1, 2))
    tversky = 1 - ((true_pos + 1) / (true_pos + alpha * false_neg + beta * false_pos + 1)).mean()

    # tversky controls how wide the crack comes out but says nothing about whether it stays in one
    # piece, and the u-net was scoring worse on boundary f1 than the map it learned from. cldice
    # compares centrelines, so a mask that is split into fragments loses even if the overlap is fine
    topology = 1 - cl_dice(probability.unsqueeze(1), reference.unsqueeze(1))
    return ce + tversky + W_CLDICE * topology


print(f"U-Net parameters: {sum(p.numel() for p in UNet(pretrained=False).parameters()) / 1e6:.1f}M")


In [ ]:
SEG_CKPT = WORK_DIR / "unet.pth"
CONFIDENCE = 0.7


def make_seg_loader(frame, pseudo_dir, transform, shuffle):
    return DataLoader(
        PseudoMaskDataset(frame, pseudo_dir, transform),
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2,
        pin_memory=torch.cuda.is_available(), drop_last=shuffle,
    )


def run_seg_epoch(unet, loader, optimizer=None, scheduler=None, scaler=None):
    training = optimizer is not None
    unet.train(training)
    total = 0.0
    for batch in tqdm(loader, desc="train" if training else "val", leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["target"].to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training), torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            loss = pseudo_loss(unet(images), targets)
        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(unet.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        total += float(loss.detach())
    return total / len(loader)


@torch.no_grad()
def relabel_with_unet(unet, frame, out_dir):
    """Self-training, the U-Net relabels the data. The image level gate still applies."""
    out_dir.mkdir(parents=True, exist_ok=True)
    unet.eval()
    for batch in tqdm(make_loader(frame, eval_tf), desc=f"re-labelling -> {out_dir.name}", leave=False):
        probability = torch.sigmoid(unet(batch["image"].to(DEVICE))).squeeze(1).float().cpu().numpy()
        for i, stem in enumerate(batch["stem"]):
            if batch["label"][i] < 0.5:
                label = np.zeros(probability[i].shape, np.uint8)     # negative image, always fully background
            else:
                # cap the area as well, otherwise every round hands the next one a slightly
                # thicker crack and the thickening compounds
                high = max(CONFIDENCE, area_threshold(probability[i], RELABEL_AREA_CAP))
                label = np.full(probability[i].shape, IGNORE_INDEX, np.uint8)
                label[probability[i] <= 1 - CONFIDENCE] = 0
                label[probability[i] >= high] = 1
            Image.fromarray(label).save(out_dir / f"{stem}.png")


set_seed()
unet = UNet().to(DEVICE)
seg_history, pseudo_dir = [], PSEUDO_DIR
ROUND_CKPT = WORK_DIR / "unet_round.pth"

# each round trains against its own pseudo-masks, so the per-round validation losses are computed
# against different targets and cannot be compared. this loader keeps the stage-2 masks fixed and
# gives every round one number on the same scale, which is how we pick the model we ship
reference_loader = make_seg_loader(val_df, PSEUDO_DIR / "val", seg_eval_tf, shuffle=False)
best_reference = float("inf")

for round_id in range(1, SELF_TRAINING_ROUNDS + 1):
    train_seg_loader = make_seg_loader(train_df, pseudo_dir / "train", seg_train_tf, shuffle=True)
    val_seg_loader = make_seg_loader(val_df, pseudo_dir / "val", seg_eval_tf, shuffle=False)

    encoder_params = [p for n, p in unet.named_parameters() if n.startswith(("stem", "layer"))]
    decoder_params = [p for n, p in unet.named_parameters() if not n.startswith(("stem", "layer"))]
    optimizer = torch.optim.AdamW(
        [{"params": encoder_params, "lr": 3e-5}, {"params": decoder_params, "lr": 3e-4}], weight_decay=1e-4)
    scheduler = cosine_with_warmup(optimizer, SEG_EPOCHS * len(train_seg_loader), warmup=100)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best_val, stale = float("inf"), 0

    for epoch in range(1, SEG_EPOCHS + 1):
        started = time.time()
        train_loss = run_seg_epoch(unet, train_seg_loader, optimizer, scheduler, scaler)
        val_loss = run_seg_epoch(unet, val_seg_loader)
        seg_history.append({"round": round_id, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"round {round_id} epoch {epoch:2d}  train {train_loss:.4f}  val {val_loss:.4f}  ({time.time() - started:.0f}s)")
        if val_loss < best_val - 1e-4:              # selection on pseudo-masks, not ground truth
            best_val, stale = val_loss, 0
            torch.save(unet.state_dict(), ROUND_CKPT)
        else:
            stale += 1

        if epoch >= MIN_EPOCHS and stale >= SEG_PATIENCE:
            print(f"round {round_id} has not improved for {SEG_PATIENCE} epochs, stopping")
            break

    unet.load_state_dict(torch.load(ROUND_CKPT, map_location=DEVICE))
    reference_loss = run_seg_epoch(unet, reference_loader)
    print(f"round {round_id} scores {reference_loss:.4f} against the stage-2 masks")
    if reference_loss < best_reference:
        best_reference = reference_loss
        torch.save(unet.state_dict(), SEG_CKPT)

    if round_id < SELF_TRAINING_ROUNDS:
        pseudo_dir = SCRATCH_DIR / f"pseudo_round{round_id + 1}"
        for split, frame in [("train", train_df), ("val", val_df)]:
            relabel_with_unet(unet, frame, pseudo_dir / split)

# a later round is allowed to be worse than an earlier one, and in that case we keep the earlier one
unet.load_state_dict(torch.load(SEG_CKPT, map_location=DEVICE))
seg_history = pd.DataFrame(seg_history)
stage_done("stage 3, u-net self-training")


## 5. Results

Everything above is frozen now. The test split was never seen by any training loop, any threshold or
any model selection decision, so we can finally open its masks.

Every variant gives a probability map at 448x448 which we upsample to the mask's own resolution
before thresholding, so all of them are scored on the same pixels. The gate at test time uses the
classifier's prediction and not the true bit, because a deployed model would not have the bit
either.

The CAM variants binarise with a per image quantile, so leaving the U-Net on a flat 0.5 was
comparing two different rules. The U-Net threshold is now picked on validation as the value whose
average predicted area matches `TOPK_RATIO`. That is the same size prior everything else uses and
it needs the images only, no masks, so it is a legal choice and the same one a deployed model could
make. The sweep in section 6 shows what an oracle threshold would have bought instead.

| Metric | What it tells us |
|---|---|
| IoU / Dice (over the whole dataset) | Overall pixel agreement, pooled over all images. |
| Mean image IoU | The same but averaged per image over the cracked ones, so a few huge cracks do not dominate. |
| Precision / recall | Whether we over or under segment. |
| Boundary F1 (+-2 px) | Region metrics are dominated by area, which is harsh for something a few pixels wide. This asks whether the contour is roughly in the right place. |

The four variants below separate the contribution of each idea: the raw CAM, the TTA, the guided
filter, and the U-Net distillation with self-training.

In [ ]:
def boundary_f1(prediction, target, tolerance=2):
    """F1 between predicted and true contours, allowing a few pixels of slack."""
    if prediction.sum() == 0 and target.sum() == 0:
        return 1.0
    if prediction.sum() == 0 or target.sum() == 0:
        return 0.0
    kernel = np.ones((3, 3), np.uint8)
    slack = np.ones((2 * tolerance + 1, 2 * tolerance + 1), np.uint8)
    pred_edge = cv2.morphologyEx(prediction, cv2.MORPH_GRADIENT, kernel)
    true_edge = cv2.morphologyEx(target, cv2.MORPH_GRADIENT, kernel)
    precision = (pred_edge * cv2.dilate(true_edge, slack)).sum() / max(pred_edge.sum(), 1)
    recall = (true_edge * cv2.dilate(pred_edge, slack)).sum() / max(true_edge.sum(), 1)
    return float(2 * precision * recall / max(precision + recall, 1e-9))


@torch.no_grad()
def unet_probability(unet, images):
    """U-Net probability averaged over the four flips, the same trick the CAM already gets."""
    unet.eval()
    total = torch.zeros(images.shape[0], images.shape[2], images.shape[3], device=images.device)
    for dims in [(), (-1,), (-2,), (-1, -2)]:
        flipped = torch.flip(images, dims) if dims else images
        logits = unet(flipped)
        if dims:
            logits = torch.flip(logits, dims)
        total += torch.sigmoid(logits).squeeze(1).float()
    return total / 4


@torch.no_grad()
def calibrate_unet_threshold(frame, target_area=TOPK_RATIO):
    """Pixel threshold chosen so the predicted area matches the size prior. No masks."""
    values = []
    for batch in tqdm(make_loader(sample_n(frame, 300), eval_tf), desc="calibrating", leave=False):
        probability = unet_probability(unet, batch["image"].to(DEVICE)).cpu().numpy()
        # every fourth pixel is plenty for a quantile and keeps this in memory
        values.extend(probability[i].ravel()[::4] for i in range(len(probability)) if batch["label"][i] > 0.5)
    if not values:
        return 0.5
    # a quantile over the pooled probabilities hits the target area exactly, where a search over a
    # grid of thresholds cannot: the U-Net output is bimodal, so 2% of the pixels can easily all sit
    # above 0.95 and every candidate on the grid then gives the same answer
    return float(np.clip(np.quantile(np.concatenate(values), 1.0 - target_area), 0.05, 0.99))


# the CAM variants binarise with a per-image quantile, so leaving the U-Net on a flat 0.5 compared
# two different rules. this puts both on the same size prior, and it is still label-free
UNET_THRESHOLD = calibrate_unet_threshold(val_df)
print(f"u-net threshold picked on val: {UNET_THRESHOLD:.2f} (target area {TOPK_RATIO:.1%})")


@torch.no_grad()
def predict_probability(variant, batch):
    """Probability map at IMG_SIZE + the image-level gate from the classifier."""
    images = batch["image"].to(DEVICE)
    gate = (torch.sigmoid(model(images)["logit"].squeeze(1)).cpu().numpy() >= GATE_THRESHOLD)

    if variant == "cam":
        probability = multi_scale_cam(model, images, scales=(1.0,), flips=False)[:, 0].cpu().numpy()
    elif variant == "cam_tta":
        probability = multi_scale_cam(model, images)[:, 0].cpu().numpy()
    elif variant == "cam_tta_gf":
        cams = multi_scale_cam(model, images)[:, 0].cpu().numpy()
        probability = np.stack([refined_probability(cams[i], batch["raw"][i].numpy()) for i in range(len(cams))])
    elif variant == "unet":
        probability = unet_probability(unet, images).cpu().numpy()
    else:
        raise ValueError(variant)
    return probability, gate


def binarise(probability, variant):
    """Same size prior for both: a per-image quantile for the CAM, a calibrated one for the U-Net."""
    if variant == "unet":
        return (probability >= UNET_THRESHOLD).astype(np.uint8)
    return (probability >= area_threshold(probability)).astype(np.uint8)


def evaluate(variant, frame=None, tag=None):
    frame = test_df if frame is None else frame
    loader = make_loader(frame, eval_tf, with_raw=True)
    tp = fp = fn = tn = 0
    image_iou, boundary = [], []

    for batch in tqdm(loader, desc=f"evaluating {variant}", leave=False):
        probability, gate = predict_probability(variant, batch)
        for i, stem in enumerate(batch["stem"]):
            truth = load_gt_mask(stem)
            prob = cv2.resize(probability[i], (truth.shape[1], truth.shape[0]), interpolation=cv2.INTER_LINEAR)
            prediction = binarise(prob, variant) if gate[i] else np.zeros_like(truth)

            p, t = prediction.astype(bool), truth.astype(bool)
            i_tp, i_fp, i_fn = int((p & t).sum()), int((p & ~t).sum()), int((~p & t).sum())
            tp, fp, fn, tn = tp + i_tp, fp + i_fp, fn + i_fn, tn + int((~p & ~t).sum())
            if t.any():
                image_iou.append(i_tp / max(i_tp + i_fp + i_fn, 1))
                boundary.append(boundary_f1(prediction, truth))

    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {
        "variant": tag or variant,
        "IoU": tp / max(tp + fp + fn, 1),
        "Dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "precision": precision,
        "recall": recall,
        "mean image IoU": float(np.mean(image_iou)) if image_iou else 0.0,
        "boundary F1": float(np.mean(boundary)) if boundary else 0.0,
        "pixel acc": (tp + tn) / max(tp + tn + fp + fn, 1),
    }


set_seed()
assert GT_READS["train"] == 0, f"a training mask was opened {GT_READS['train']} times"
print(f"supervision audit: {GT_READS['train']} training masks opened, "
      f"{GT_READS['held out']} held-out masks read so far, and only for figures")

results = pd.DataFrame([
    evaluate("cam", tag="1. CAM (single scale)"),
    evaluate("cam_tta", tag="2. + multi-scale/flip TTA"),
    evaluate("cam_tta_gf", tag="3. + guided-filter refinement"),
    evaluate("unet", tag=f"4. + U-Net distillation ({SELF_TRAINING_ROUNDS} rounds)"),
]).set_index("variant")

display(results.round(4))
stage_done("evaluation")


In [ ]:
seg_history = pd.DataFrame(seg_history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

x = np.arange(len(results))
for offset, metric, colour in [(-0.2, "IoU", "#3182bd"), (0.0, "Dice", "#e6550d"), (0.2, "boundary F1", "#31a354")]:
    axes[0].bar(x + offset, results[metric], width=0.2, label=metric, color=colour)
axes[0].set_xticks(x, [f"{i + 1}" for i in range(len(results))])
axes[0].set_xlabel("ablation step")
axes[0].set_title("Ablation on the test split")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].plot(results.recall, results.precision, "o-", color="#756bb1")
for name, row in results.iterrows():
    axes[1].annotate(name.split(".")[0], (row.recall, row.precision), textcoords="offset points", xytext=(6, 4))
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("Precision / recall trade-off")
axes[1].grid(alpha=0.3)

for round_id, group in seg_history.groupby("round"):
    steps = np.arange(len(group)) + (round_id - 1) * SEG_EPOCHS
    axes[2].plot(steps, group.train_loss, label=f"round {round_id} train")
    axes[2].plot(steps, group.val_loss, "--", label=f"round {round_id} val")
axes[2].set_xlabel("epoch")
axes[2].set_title("U-Net loss on pseudo-masks")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(results[["IoU", "Dice", "boundary F1"]].round(4).to_string())


In [ ]:
@torch.no_grad()
def show_predictions(frame, title):
    frame = frame.reset_index(drop=True)
    batch = next(iter(make_loader(frame, eval_tf, batch_size=len(frame), with_raw=True)))
    images = batch["image"].to(DEVICE)
    cam = multi_scale_cam(model, images)[:, 0].cpu().numpy()
    unet_prob = torch.sigmoid(unet(images)).squeeze(1).float().cpu().numpy()

    columns = ["image", "CAM (stage 2)", "U-Net probability", "U-Net mask", "ground truth", "error map"]
    fig, axes = plt.subplots(len(frame), 6, figsize=(17, 2.85 * len(frame)), squeeze=False)
    for i, stem in enumerate(batch["stem"]):
        truth = load_gt_mask(stem, (IMG_SIZE, IMG_SIZE))
        prediction = (unet_prob[i] >= UNET_THRESHOLD).astype(np.uint8)
        error = np.stack([prediction & (1 - truth), prediction & truth, truth & (1 - prediction)], -1).astype(float)

        for j, panel in enumerate([batch["raw"][i].numpy(), cam[i], unet_prob[i], prediction, truth, error]):
            ax = axes[i][j]
            if j in (1, 2):
                ax.imshow(panel, cmap="inferno", vmin=0, vmax=1)
            elif j in (3, 4):
                ax.imshow(panel, cmap="gray", vmin=0, vmax=1)
            else:
                ax.imshow(panel)
            if i == 0:
                ax.set_title(columns[j], fontsize=11)
            ax.axis("off")
    fig.suptitle(f"{title}   ·   error map: green = correct, red = false positive, blue = missed", y=1.005)
    plt.tight_layout()
    plt.show()


# everything below runs on validation. the test split is scored exactly once, by the table above,
# and looking through test images to find the interesting ones would quietly undo that
positives = val_df[val_df.label == 1]
show_predictions(sample_n(positives, 5), "Typical predictions (validation)")

# failure analysis: rank a sample of cracked images by how badly the U-Net disagrees with the truth
scan = sample_n(positives, 200)
scores = []
for _, row in tqdm(scan.iterrows(), total=len(scan), desc="scanning for failures"):
    image = eval_tf(image=np.asarray(Image.open(row.image_path).convert("RGB")))["image"][None].to(DEVICE)
    with torch.no_grad():
        prediction = (torch.sigmoid(unet(image))[0, 0].float().cpu().numpy() >= UNET_THRESHOLD).astype(np.uint8)
    truth = load_gt_mask(row.stem, (IMG_SIZE, IMG_SIZE))
    scores.append((int((prediction & truth).sum()) / max(int((prediction | truth).sum()), 1), row.stem))

worst = [stem for _, stem in sorted(scores)[:5]]
show_predictions(val_df[val_df.stem.isin(worst)], f"Worst {len(worst)} of {len(scan)} sampled cracked images")


In [ ]:
# The sweep below is an oracle curve: picking its best point would be choosing a hyperparameter
# with masks. It runs on validation and it is only here to show how much the mask-free
# calibrated threshold gives up against the best one could have done knowing the answer.
thresholds = np.linspace(0.05, 0.95, 19)
sweep_tp = np.zeros_like(thresholds)
sweep_fp = np.zeros_like(thresholds)
sweep_fn = np.zeros_like(thresholds)
per_source = {}

sample_df = val_df.sample(min(400, len(val_df)), random_state=SEED).reset_index(drop=True)
for batch in tqdm(make_loader(sample_df, eval_tf, with_raw=True), desc="threshold sweep", leave=False):
    probability, gate = predict_probability("unet", batch)
    for i, stem in enumerate(batch["stem"]):
        truth = load_gt_mask(stem, (IMG_SIZE, IMG_SIZE)).astype(bool)
        prob = probability[i] * gate[i]
        for k, threshold in enumerate(thresholds):
            prediction = prob >= threshold
            hit = int((prediction & truth).sum())
            sweep_tp[k] += hit
            sweep_fp[k] += int(prediction.sum()) - hit
            sweep_fn[k] += int(truth.sum()) - hit
        if truth.any():
            prediction = prob >= UNET_THRESHOLD
            hit = int((prediction & truth).sum())
            group = stem.split("_")[0].lower()
            per_source.setdefault(group, []).append(hit / max(int((prediction | truth).sum()), 1))

sweep_iou = sweep_tp / np.maximum(sweep_tp + sweep_fp + sweep_fn, 1)
source_iou = pd.Series({k: np.mean(v) for k, v in per_source.items()}).sort_values()
source_n = pd.Series({k: len(v) for k, v in per_source.items()})

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(thresholds, sweep_iou, "o-")
axes[0].axvline(UNET_THRESHOLD, color="crimson", ls="--", label=f"mask-free choice ({UNET_THRESHOLD:.2f})")
axes[0].scatter([thresholds[sweep_iou.argmax()]], [sweep_iou.max()], color="green", zorder=5,
                label=f"oracle best = {sweep_iou.max():.3f} @ {thresholds[sweep_iou.argmax()]:.2f}")
axes[0].set_xlabel("binarisation threshold")
axes[0].set_ylabel("dataset IoU")
axes[0].set_title("How much does the threshold matter?")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].barh(source_iou.index, source_iou.values, color="#3182bd")
for i, (name, value) in enumerate(source_iou.items()):
    axes[1].text(value, i, f"  {value:.2f}  (n={source_n[name]})", va="center", fontsize=8)
axes[1].set_xlim(0, min(1.0, source_iou.max() * 1.45))
axes[1].set_xlabel("mean image IoU")
axes[1].set_title("Where does it work? (per source sub-dataset)")
plt.tight_layout()
plt.show()

print(f"oracle would reach IoU {sweep_iou.max():.4f} at {thresholds[sweep_iou.argmax()]:.2f}, "
f"the mask-free threshold gets {sweep_iou[np.argmin(np.abs(thresholds - UNET_THRESHOLD))]:.4f}")


## 6. Analysis and conclusions

### What each step gave us

Going down the ablation table, one change per row:

1. Single scale CAM is the baseline. The location is right, the boundaries are not. At stride 8 a
   3 pixel crack is a fraction of one cell.
2. Multi-scale and flip TTA costs no extra training, only around 12x the inference time, and the
   large scale is what recovers the thin structure. The U-Net gets a cheaper 4x flip average at inference.
3. The guided filter is where the map stops looking like a blob. It is also the step where most
   people would put a domain specific edge detector, and using a generic one is what keeps the
   pipeline reusable. It reliably buys boundary F1, which is what it is for. Whether it also buys
   IoU is less clear: it sharpens the map onto image edges, and on a textured surface some of those
   edges are not the crack, so it can trade a little area agreement for a better placed contour.
   The pseudo-masks are built from the refined map either way, because the U-Net cares much more
   about where the boundary is than about a fraction of a percent of overlap.
4. U-Net distillation with self-training is the biggest single jump, and the student ends up better
   than the teacher it was trained on. The CAM errors are inconsistent between images while the real
   signal is not, and the ignore band leaves the student room to correct the teacher.

Everything in this section runs on the validation split. The test split is scored exactly once, by
the table in section 5, and hunting through test images for the interesting ones would quietly undo
that. `load_gt_mask` counts every mask it opens and section 5 asserts that the count for the
training split is still zero.

### Where it fails

* Thin low contrast cracks on textured concrete. The classifier can be right for the wrong reason
  (the texture) and then the CAM covers a region instead of a line.
* Shadows, joints and painted lines look the same as a crack at the image level. They are most of
  our false positives and an image level label cannot tell them apart.
* Cracks touching the border are under segmented. The puzzle term helps but the border tiles still
  see the least context.
* Recall is higher than precision everywhere. With only a bag level label the model learns where the
  evidence is, and the evidence is thicker than the crack itself. The area prior threshold and the
  asymmetric Tversky loss pull it back a long way, but they do not close the gap, they only stop it
  from compounding at every stage.
* The crack free images almost all come from one source sub-dataset, so the classifier can score well
  by recognising the source instead of the crack. The heavy photometric augmentation and the balanced
  sampler are there to make that shortcut harder, and the per source table is how we check whether it
  worked.

### Compared to full supervision

A fully supervised U-Net on this dataset gets around 0.65-0.75 IoU. Getting a good part of that with
no pixel annotations at all is the point of the exercise, and the annotation cost is one click per
image instead of a traced mask.

### Using it on something else

Nothing above is crack specific. To run it on "find the tumour cells given only healthy/sick slide
labels":

1. Point `DATA_ROOT` at the new dataset and give it `image_path, label`.
2. Set `TOPK_RATIO` to a rough guess of how much of the image the target covers, and
   `W_SPARSITY = 0` if the target is large.
3. Nothing else changes.

### What we would try next

* A learned pixel affinity network (IRNet / AffinityNet) instead of the guided filter, to spread the
  CAM seeds along learned boundaries instead of image gradients.
* Contrastive scale equivariance (SEAM) next to the puzzle consistency.
* Training the classifier at output stride 4 with a bigger crop. The stride 8 map is still the
  tightest bottleneck in the whole pipeline.
* A leave one source out run: train on every sub-dataset but one and test on the one left out. The
  per source table says the classifier is right almost everywhere, but nine of the eleven sources
  contain only cracked images, so that table cannot separate "knows what a crack is" from "knows
  which folder this came from". Leave one source out is the experiment that could, and it costs one
  full training run per source, which is why it is future work and not a result.
* A domain adversarial head on the source label with a gradient reversal layer. We left it out on
  purpose: it is the standard answer to a shortcut, but there is no measured gap for it to close
  yet, and adding a regulariser against a problem we have not demonstrated would be method for its
  own sake. It belongs after the leave one source out run, not before it.

## 7. Kaggle submission

The competition test set is a separate folder of images with no masks and no labels, so here the
pipeline runs the way it would in deployment: the classifier decides whether there is a crack at all
and only then does the U-Net say where it is.

The mask has to be sent at the native resolution of each test image, so we upsample the probability
map from 448x448 before thresholding. Images the classifier calls negative get an empty string,
which is what the sample submission uses for "no crack".

In [ ]:
COMP_DIR = Path(os.environ.get("COMP_DIR", "/kaggle/input/dl-2025-project-2-pro"))
TEST_IMAGE_DIR = COMP_DIR / "test" / "images"
SAMPLE_CSV = COMP_DIR / "sample_submission.csv"

# if the competition data is not attached, fall back to the same test images in the dataset
if not TEST_IMAGE_DIR.is_dir():
    TEST_IMAGE_DIR = DATA_ROOT / "test" / "images"


# https://www.kaggle.com/paulorzp/rle-functions-run-lenght-encode-decode
def mask2rle(img):
    """Binary mask -> the run length string the competition expects."""
    pixels = img.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


def rle2mask(rle, shape):
    """Inverse of mask2rle, only used to check that the encoding round-trips."""
    mask = np.zeros(shape[0] * shape[1], np.uint8)
    if isinstance(rle, str) and rle.strip():
        numbers = [int(x) for x in rle.split()]
        for start, length in zip(numbers[::2], numbers[1::2], strict=True):
            mask[start - 1:start - 1 + length] = 1
    return mask.reshape(shape[1], shape[0]).T


check = np.zeros((40, 30), np.uint8)
check[5:12, 8:20] = 1
check[30, :] = 1
assert (rle2mask(mask2rle(check), check.shape) == check).all()
assert mask2rle(np.zeros((10, 10), np.uint8)) == ""
print("rle round-trip ok")

In [ ]:
class CompetitionTestSet(Dataset):
    """Test images only - no mask, no label, we resize and remember the original size."""

    def __init__(self, files, image_dir, transform):
        self.files = list(files)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name = self.files[idx]
        image = np.asarray(Image.open(self.image_dir / name).convert("RGB"))
        height, width = image.shape[:2]
        return {"image": self.transform(image=image)["image"], "file": name, "height": height, "width": width}


@torch.no_grad()
def predict_test_masks(files, batch_size=BATCH_SIZE):
    """Classifier gate first, then the U-Net mask at the image's own resolution."""
    model.eval()
    unet.eval()
    loader = DataLoader(
        CompetitionTestSet(files, TEST_IMAGE_DIR, seg_eval_tf),
        batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available(),
    )

    masks = {}
    for batch in tqdm(loader, desc="predicting the test set"):
        images = batch["image"].to(DEVICE)
        gate = torch.sigmoid(model(images)["logit"].squeeze(1)).cpu().numpy() >= GATE_THRESHOLD
        probability = unet_probability(unet, images).cpu().numpy()

        for i, name in enumerate(batch["file"]):
            height, width = int(batch["height"][i]), int(batch["width"][i])
            if not gate[i]:
                masks[name] = np.zeros((height, width), np.uint8)
                continue
            resized = cv2.resize(probability[i], (width, height), interpolation=cv2.INTER_LINEAR)
            masks[name] = (resized >= UNET_THRESHOLD).astype(np.uint8)
    return masks

In [ ]:
if not TEST_IMAGE_DIR.exists():
    print(f"{TEST_IMAGE_DIR} not found - attach the competition data to the notebook to submit")
else:
    # sample_submission.csv only has a few example rows, so all we take from it is the column
    # names and the fact that the file column keeps the .jpg extension
    if SAMPLE_CSV.exists():
        print(pd.read_csv(SAMPLE_CSV).head(3).to_string(max_colwidth=40))

    test_files = sorted(p.name for p in TEST_IMAGE_DIR.iterdir() if p.is_file())
    print(f"\n{len(test_files)} test images")

    set_seed()
    test_masks = predict_test_masks(test_files)

    submission = pd.DataFrame({
        "file": test_files,
        "crack": [mask2rle(test_masks[name]) for name in test_files],
    })
    submission.to_csv(WORK_DIR / "submission.csv", index=False)

    positives = int((submission.crack.str.len() > 0).sum())
    coverage = float(np.mean([m.mean() for m in test_masks.values()]))
    print(f"wrote {WORK_DIR / 'submission.csv'}")
    print(f"{positives}/{len(submission)} images predicted as cracked ({positives / len(submission):.1%})")
    print(f"average predicted crack area: {coverage:.2%} of the image")
    stage_done("submission")

In [ ]:
# eyeball a few test predictions - there is no ground truth here, so this is all we can check
if TEST_IMAGE_DIR.exists():
    preview = list(test_masks)[:6]
    fig, axes = plt.subplots(2, len(preview), figsize=(2.8 * len(preview), 6))
    for j, name in enumerate(preview):
        axes[0][j].imshow(Image.open(TEST_IMAGE_DIR / name).convert("RGB"))
        axes[0][j].set_title(name, fontsize=8)
        axes[1][j].imshow(test_masks[name], cmap="gray", vmin=0, vmax=1)
        for row in (0, 1):
            axes[row][j].axis("off")
    plt.tight_layout()
    plt.show()

## References

Methods, in the order they appear in the notebook.

* Jo, S. and Yu, I.-J. (2021). *Puzzle-CAM: improved localization via matching partial and full
  features.* ICIP. The tiled consistency term in stage 1.
* Zhou, Z.-H. (2004). *Multi-instance learning: a survey.* And Ilse, M., Tomczak, J. and Welling, M.
  (2018), *Attention-based deep multiple instance learning*, ICML, for the bag/instance framing and
  the case for pooling that is neither mean nor max.
* Zhou, B. et al. (2016). *Learning deep features for discriminative localization.* CVPR. Class
  activation maps, and the reason the 1x1 head carries no bias.
* He, K., Sun, J. and Tang, X. (2013). *Guided image filtering.* TPAMI. The edge preserving filter in
  stage 2.
* Salehi, S. S. M., Erdogmus, D. and Gholipour, A. (2017). *Tversky loss function for image
  segmentation using 3D fully convolutional deep networks.* MLMI. The asymmetric region loss.
* Shit, S. et al. (2021). *clDice - a novel topology-preserving loss function for tubular structure
  segmentation.* CVPR. The soft skeleton and the centreline loss.
* Ronneberger, O., Fischer, P. and Brox, T. (2015). *U-Net: convolutional networks for biomedical
  image segmentation.* MICCAI.
* He, K. et al. (2016). *Deep residual learning for image recognition.* CVPR. The ResNet-34 encoder,
  ImageNet weights from torchvision.
* Chen, L.-C. et al. (2017). *Rethinking atrous convolution for semantic image segmentation.* The
  dilated trunk that keeps the score map at stride 8.
* Ahn, J., Cho, S. and Kwak, S. (2019). *Weakly supervised learning of instance segmentation with
  inter-pixel relations.* CVPR. IRNet, listed above as the alternative to the guided filter.
* Wang, Y. et al. (2020). *Self-supervised equivariant attention mechanism for weakly supervised
  semantic segmentation.* CVPR. SEAM, listed above as future work.
* Dataset: [Crack Segmentation Dataset](https://www.kaggle.com/datasets/lakshaymiddha/crack-segmentation-dataset),
  a merge of CFD, DeepCrack, Rissbilder, GAPs, CrackTree, Volker, Eugen Muller, Sylvie Chambon,
  forest and a set of crack free surfaces.
